# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [ ]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder
from preprocessing.metrics.reading.words import all_tokens_from_aois
from pymovements.measure.reading.processing import compute_reading_measures

import polars as pl

In [ ]:
# If you have a specific config file, load it here:
# settings.load_from_yaml("data/MultiplEYE_<...>/multipleye_settings_preprocessing.yaml")

In [ ]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [ ]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Overwrite:       {settings.OVERWRITE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [ ]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [ ]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [ ]:
preprocessing.run_preflight_check(multipleye)

## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [ ]:
multipleye.convert_edf_to_asc()

Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [ ]:
multipleye.prepare_session_level_information()

In [ ]:
# print an overview on the data collection and the sessions
multipleye

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [ ]:
# pick only one session as an example to work with in the next steps
sessions = list(multipleye)  # list of Session objects
sess = sessions[0]  # a real Session
sid = sess.sid  # get the session ID (Sid) from the Session
sid

In [ ]:
type(sess)

### Creating Gaze Frame from ASCII File

In [ ]:
gaze = preprocessing.load_gaze_data(
    asc_file=sess.asc_path,
    lab_config=sess.lab_config,
    sid=sess.sid,
    trial_cols=settings.TRIAL_COLS,
    messages=settings.ANSWER_MSG_PATTERNS,  # for comprehension questions later - see Stage 4.
)

In [ ]:
# save gaze and metadata
preprocessing.save_raw_data(sid, gaze)
preprocessing.save_session_metadata(sid, gaze)

In [ ]:
sid.raw_data_dir, sid.metadata_dir

### Output directory structure

All preprocessed data is organised by data type under `preprocessed_data/<data_collection_name>/`. Each data type folder contains one subfolder per session:

```
preprocessed_data/<dcn>/
├── raw_data/
│   └── <session_save_name>/
├── fixations/
│   └── <session_save_name>/
├── saccades/
│   └── <session_save_name>/
├── scanpaths/
│   └── <session_save_name>/
├── reading_measures/
│   └── <session_save_name>/
├── sanity_checks/
│   └── <session_save_name>/
├── metadata/
│   └── <session_save_name>/
│       ├── gaze_metadata.json
│       ├── experiment.yaml
│       ├── calibrations.tsv
│       ├── calibrations.feather
│       ├── validations.tsv
│       ├── validations.feather
│       └── <session_idf>_overview.yaml
├── participant_data.csv
├── <dcn>_overview.yaml
└── stimuli_<dcn>/
```

The `Sid` object provides convenient properties to access each path:

In [ ]:
print(f"Raw data:        {sid.raw_data_dir}")
print(f"Metadata:        {sid.metadata_dir}")
print(f"Fixations:       {sid.fixations_dir}")
print(f"Saccades:        {sid.saccades_dir}")
print(f"Scanpaths:       {sid.scanpaths_dir}")
print(f"Reading measures: {sid.reading_measures_dir}")

In order to have the metadata which is extracted by pymovements available to create out session overview, we get this information from pymovements and store it in our session object.

In [ ]:
sess.pm_gaze_metadata = gaze._metadata
sess.calibrations = gaze.calibrations
sess.validations = gaze.validations

### Coordinate and Velocity Preprocessing

Eye movements are recorded in screen pixel coordinates, which depend on stimulus size and monitor setup. To compare gaze behavior across participants, screens, or datasets, it is standard to convert pixel positions 
into **degrees of visual angle (dva)**. Next, we compute **gaze velocity**, which allows us to detect saccades and distinguish them from fixations.

In [ ]:
# inspect the gaze samples
gaze.samples.head()

In [ ]:
preprocessing.preprocess_gaze(gaze)

In [ ]:
# inspect the preprocessed gaze samples, the dataframe should now also contain a position in dva and velocity columns
gaze.samples.head()

Save our events data.

In [ ]:
gaze = preprocessing.load_trial_level_events_data(
    gaze,
    sess.sid,
    event_type=settings.FIXATION,
    file_pattern=None,
)

gaze = preprocessing.load_trial_level_events_data(
    gaze,
    sess.sid,
    event_type=settings.SACCADE,
    file_pattern=None,
)

## Stage 2b: Map Fixations to AOIs

Once we have the fixations, we can map each of them to the AOIs of the stimulus. The resulting scanpath can then be saved. Note that this features is not yet completely finished.

In [ ]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [ ]:
# The resulting mapping can be stored as a scanpath, which is a sequence of AOIs that were fixated in the order they were fixated.
preprocessing.save_scanpaths(sid, gaze)

In [ ]:
# save metadata again
preprocessing.save_session_metadata(sid, gaze)

## NEW

In [ ]:
group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]

only_fix = (
    gaze.events.frame.filter(
        (pl.col("name") == settings.FIXATION)
        & (pl.col(settings.WORD_IDX_COL).is_not_null())
    )
    .with_row_count("fixation_id")
    .sort(group_columns + ["onset"])
)
rm_all_trials = []

In [ ]:
stim = sess.stimuli[0]
aois = stim.text_stimulus.aois
words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
words_only = words_only.with_columns(pl.lit(stim.name).alias(settings.STIMULUS_COL))
trial_idx = stim.trial_id

In [ ]:
trial_fix = only_fix.filter((pl.col(settings.TRIAL_COL) == trial_idx))

In [ ]:
grouped_fix = trial_fix.group_by(settings.PAGE_COL)

In [ ]:
grouped_fix.head()

In [ ]:
settings.PAGE_PREFIX

In [ ]:
grouped_fix.head()

In [ ]:
for page in grouped_fix:
    print(page)

In [ ]:
aois.group_by(settings.PAGE_COL).head()

In [ ]:
words_only

In [ ]:
for fix in only_fix.group_by(group_columns):
    print(fix)

In [ ]:
for (trial_idx, stim_idx, page_idx), df in only_fix.group_by(group_columns):
    words_only = all_tokens_from_aois(df, trial=trial_idx)
    print(words_only.head())

In [ ]:
stimuli = sess.stimuli
stim = stimuli[0]
stim

In [ ]:
aois = stim.text_stimulus.aois
words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
words_only = words_only.with_columns(pl.lit(stim.name).alias(settings.STIMULUS_COL))
trial_idx = stim.trial_id

In [ ]:
words_only

In [ ]:
words_only_all_trials = []
for stim in stimuli:
    aois = stim.text_stimulus.aois
    words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
    words_only = words_only.with_columns(pl.lit(stim.name).alias("stimulus"))
    words_only_all_trials.append(words_only)

words_df = pl.concat(words_only_all_trials)

In [ ]:
words_df

In [ ]:
words_df

In [ ]:
only_fix

In [ ]:
words_df

In [ ]:
settings.STIMULUS_COL

In [ ]:
words_df

In [ ]:
for (trial_idx, stim_name, page_idx), fix_df in only_fix.group_by(group_columns):
    print(stim_name)
    print(words_df.filter((pl.col(settings.STIMULUS_COL) == stim_name)))

In [ ]:
rm_all_trials = []

for (trial_idx, stim_name, page_idx), fix_df in only_fix.group_by(group_columns):
    page_words = words_df.filter(
        (pl.col(settings.TRIAL_COL) == trial_idx)
        & (pl.col(settings.PAGE_COL) == page_idx)
    )
    rm = compute_reading_measures(
        fixations=fix_df,
        aois=page_words,
        word_index_column=settings.WORD_IDX_COL,
        # to be replaced when words change to unit of analysis
        word_column="word",
    )
    rm = rm.with_columns(
        pl.lit(trial_idx).alias(settings.TRIAL_COL),
        pl.lit(page_idx).alias(settings.PAGE_COL),
        pl.lit(stim_name).alias(settings.STIMULUS_COL),
    )
    rm_all_trials.append(rm)

In [ ]:
rm_df = pl.concat(rm_all_trials)
rm_df

## Stage 3: Calculate AOI-based Measures

In this last step, we calculate the aoi-based measures. These are also refered to as reading measures, as they are typically used in reading research. They include measures such as first pass fixation duration (FPF), total fixation count (TFC), regression path duration (RPD), and many more. These measures are calculated based on the fixations that were mapped to the AOIs in the previous step.

### Fixation-based Metrics

As an intermediate step, the fixations are annotated. These annoataions include:
- The run ID. This ID specifies continuous sequences of fixations on the same word. It is used to calculate first pass and second pass measures.
- Whether the fixation is within the first pass or not
- The index of the preceding word and the following word
- If the saccade entering or leaving the fixation is a regression or not
- Whether it is the first fixation on the word or not

This information is necessary to calculate the reading measures in the next step.

In [ ]:
rm_df = preprocessing.calculate_reading_measures(gaze, sess.stimuli)
preprocessing.save_reading_measures(sid, rm_df)

In [ ]:
rm_df.filter(pl.col("trial") == "trial_1").head()

## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [ ]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)
parsed_answers = preprocessing.parse_answers_from_messages(gaze.messages)
source = "asc"

In [ ]:
parsed_answers

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

## Final Steps

In the very end, we can create the session and dataset overview and store them as well. In addition, the participant data can be parsed and stored.

For the MultiplEYE data, there is also the option to create a sanity check report.

In [ ]:
multipleye.create_sanity_check_report(
    gaze,
    sess.session_identifier,
    output_dir=settings.OUTPUT_DIR,
    plotting=True,
    overwrite=True,
)

In [ ]:
multipleye.create_session_overview(sess.session_identifier, path=settings.OUTPUT_DIR)
multipleye.create_dataset_overview(path=settings.OUTPUT_DIR)
multipleye.parse_participant_data(settings.OUTPUT_DIR / "participant_data.csv")

In [ ]:
from preprocessing.psychometric_tests.preprocess_psychometric_tests import (
    preprocess_all_sessions,
)

preprocess_all_sessions()